<a href="https://colab.research.google.com/github/AtacZy/Projects/blob/main/%D0%92%D1%8B%D0%B3%D1%80%D1%83%D0%B7%D0%BA%D0%B0_%D0%B0%D0%BA%D1%82%D0%B8%D0%B2%D0%BD%D0%BE%D1%81%D1%82%D0%B8_IT_Resume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **2 кейс**

**Выгрузка активности с ItResume**

**Важно**

Перед началом решения выполните следующую ячейку, чтобы загрузить необходимый для работы файл.

In [ ]:
!wget https://gist.github.com/Vs8th/a7a7f00e6cdef1b3fe87e4d61ca56e5f/raw/codesubmit.csv

In [2]:
import pandas as pd

df = pd.read_csv('codesubmit.csv', sep = ';')
df


,created_at,user_id,problem_id,is_correct,type
0,2023-04-30 13:47:14.344471,7,870,1.0,submit
1,2023-04-30 13:46:15.949925,7,870,0.0,submit
2,2023-04-30 16:13:26.005286,173,21,1.0,submit
3,2023-04-30 16:13:06.739782,173,21,NaN,run
4,2023-04-30 15:52:00.195532,173,25,1.0,submit
...,...,...,...,...,...
4994,2023-04-30 21:52:00.269123,13493,435,NaN,run
4995,2023-04-30 21:51:01.094234,13493,435,1.0,submit
4996,2023-04-30 21:50:52.059690,13493,435,NaN,run
4997,2023-04-30 21:42:24.323689,13493,1086,NaN,run


### **Решения**

#### **Задача 1**

Ваша задача - выяснить сколько в среднем тратится времени на решение задачи.

**Примечание**: для правильного подсчета - рассчитайте сначала среднее время решения по каждой задаче в отдельности, и только затем находите общее среднее время решения задач.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res`.


In [24]:
import csv
from datetime import datetime
from itertools import groupby
from collections import defaultdict

filename = 'codesubmit.csv'

# Читаем файл и помещаем файл в спиок получившиеся словари
rows = [] #Создаём список, он нужен для того то того то
with open(filename, 'r') as csvfile: # читаем файл и даём ему алиас
  reader = csv.DictReader(csvfile, delimiter=';')
  for row in reader:
    rows.append(row)

# Группируем по юзеру и задаче
grouped_data = defaultdict(dict)
for (user_id, problem_id), group in groupby(rows, lambda x: (x['user_id'], x['problem_id'])):
  for item in group:
    data_dict = {'created_at': item['created_at'], 'is_correct': item['is_correct'], 'type': item['type']}
    grouped_data[user_id].setdefault(problem_id, []).append(data_dict)

# Сортируем и фильтруем пользователей, которые решили задачи и сами задачи
result = defaultdict(dict)
for user_id, problems in grouped_data.items():
  for problem_id, submissions in problems.items():
    sorted_submissions = sorted(submissions, key=lambda x: x['created_at'], reverse=False)
    for submission in sorted_submissions:
      if submission['is_correct'] == '1':
        result[user_id][problem_id] = sorted_submissions
        break

#  Смотрим сколько времени уходит на правильное решение задачи
result1 = defaultdict(dict)
for user_id, problems in result.items():
  for problem_id, submissions in problems.items():
    time_spent = 0
    prev_time = None
    for submission in submissions:
      if submission['is_correct'] == '1':
        current_time = datetime.strptime(submission['created_at'], '%Y-%m-%d %H:%M:%S.%f')
        if prev_time:
          time_spent += round((current_time - prev_time).total_seconds(), 2)
        prev_time = None
        result1[user_id][problem_id] = time_spent
        break
      else:
        current_time = datetime.strptime(submission['created_at'], '%Y-%m-%d %H:%M:%S.%f')
        if prev_time:
          time_spent += round((current_time - prev_time).total_seconds(), 2)
        prev_time = current_time
    else:
      result1[user_id][problem_id] = time_spent

# Считаем среднее время решений по каждой проблеме причём если врменя = 0, то заносим их в отдельный список
zero_time_problems = []
time_spent = defaultdict(dict)

for user, problems in result1.items():
  for problem, time in problems.items():
    if time == 0:
      zero_time_problems.append((user, problem))
    else:
      time_spent[problem]['total_time'] = time_spent[problem].get('total_time', 0) + time
      time_spent[problem]['count'] = time_spent[problem].get('count', 0) + 1

# Считаем среднее время по каждой задаче
average_time_spent = {}
for problem, value in time_spent.items():
  average_time_spent[problem] = value['total_time'] / value['count']

avg_value = sum(average_time_spent.values()) / len(average_time_spent)
res = round(avg_value, 2)
res

611.86

✏️ ✏️ ✏️

**Проверка**

Чтобы проверить свое решение, запустите код в следующих ячейках

In [25]:
try:
    assert res == 611.86
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 2**

Ваша задача - выяснить сколько часов в среднем проводит юзер в день на платформе. Перерывы в активности за день - не учитываем.

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res2`.

**Решение**

Напишите свое решение ниже

In [33]:
import statistics
import csv
from datetime import datetime

# Читаем файл и помещаем в rows
rows = []
with open(filename, 'r') as csvfile:
  reader = csv.DictReader(csvfile, delimiter=';')
  for row in reader:
    rows.append(row)

user_atempts = {}
for atempt in rows:
  user_id = atempt['user_id']
  if user_id not in user_atempts:
    user_atempts[user_id] = []
  user_atempts[user_id].append(atempt)
  user_atempts[user_id].sort(key=lambda x: datetime.strptime(x['created_at'], '%Y-%m-%d %H:%M:%S.%f'))

user_time = {}
for user_id, attempts in user_atempts.items():
  daily_time = 0
  day_start = None
  for attempt in attempts:
    created_at = datetime.strptime(attempt['created_at'], '%Y-%m-%d %H:%M:%S.%f')
    if day_start is None:
      day_start = created_at
    daily_time += (created_at - day_start).total_seconds()
    day_start = created_at
  user_time[user_id] = daily_time/3600

res2 = round(statistics.mean(user_time.values()), 2)
res2

1.7

In [34]:
try:
    assert res2 == 1.7
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 3**

Теперь давайте посмотрим на активные сеансы. Выясните, сколько задач в среднем решается за один активный сеанс.

**Активный сеанс** - период, когда между любой активностью пользователя разница менее или равна часу, не более

Результат - число типа `float`, округлите до 2 знаков после запятой и запишите в переменную `res3`.

**Решение**

Напишите свое решение ниже

In [42]:
import csv
from datetime import datetime
from datetime import timedelta

rows = []
with open(filename, 'r') as csvfile:
  reader = csv.DictReader(csvfile, delimiter=';')
  for row in reader:
    rows.append(row)

rows.sort(key=lambda x: (x['user_id'], x['created_at']))

dates_by_user = {}
for row in rows:
  user_id = row['user_id']
  problem_id = row['problem_id']
  created_at = datetime.strptime(row['created_at'], '%Y-%m-%d %H:%M:%S.%f')
  if user_id not in dates_by_user:
    dates_by_user[user_id] = []
  dates_by_user[user_id].append((created_at, problem_id))

results = {}

for user, submissions in dates_by_user.items():
  sessions = []
  current_session = []

  for i in range(len(submissions)):
    if i == 0:
      current_session.append(submissions[i])
    else:
      time_diff = submissions[i][0] - submissions[i-1][0]
      if time_diff > timedelta(hours=1):
        sessions.append(current_session)
        current_session = [submissions[i]]
      else:
        current_session.append(submissions[i])
  # Добавили последнюю сессию
  sessions.append(current_session)

  # Посчитаем уникальные решения задач для каждой сессии
  unique_problems = []
  for session in sessions:
    for submission in session:
      problem_id = submission[1]
      if problem_id not in unique_problems:
        unique_problems.append(problem_id)
    session_count = len(unique_problems)
    unique_problems = []

  # Сохраняем результаты
    if user not in results:
      results[user] = {'sessions': 0, 'problems_solved_per_session': []}

    results[user]['sessions'] += 1
    results[user]['problems_solved_per_session'].append(session_count)

total_sessions = 0
total_problems_solved = 0

for user in results:
  sessions = results[user]['sessions']
  problems_solved = sum(results[user]['problems_solved_per_session'])

  total_sessions += sessions
  total_problems_solved += problems_solved

average_problems_solved_per_session = total_problems_solved / total_sessions

res3 = round(average_problems_solved_per_session, 2)
res3

3.14

In [43]:
try:
    assert res3 == 3.14
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!


#### **Задача 4**

И финальная - найдите самый "популярный" час дня на нашей платформе.

Популярность определяем максимальным количеством уникальных пользователей, совершающих какую-либо активность в этот период

Результат в числовом формате запишите в переменную `res4`.

Например, самым популярным часом стал период с 22 до 23, тогда в переменной `res4` должно лежать **22**. Обозначающее начало этого периода.

**Решение**

Напишите свое решение ниже

In [46]:
from datetime import datetime, timedelta

rows = []
with open(filename, 'r') as csvfile:
  reader = csv.DictReader(csvfile, delimiter=';')
  for row in reader:
    rows.append(row)

# Создаём множество уникальных значений
users = set(i['user_id'] for i in rows)

# Создадим словарь для каждого часа
activity_by_hour = {hour: 0 for hour in range(24)}

# Считаем количество уникальных пользователей в каждый час
for hour in activity_by_hour.keys():
  activity_by_hour[hour] = len(set([d['user_id'] for d in rows if datetime.strptime(d['created_at'], '%Y-%m-%d %H:%M:%S.%f').hour == hour]))

# Выводим часы с наибольшей активностью (т.е. наибольшим количеством уникальных пользователей)
max_activity_hours = [hour for hour, count in activity_by_hour.items() if count == max(activity_by_hour.values())]
res4 = max_activity_hours[0]
res4


16

In [47]:
try:
    assert res4 == 16
except AssertionError:
    print('Ответы не совпадают')
else:
    print('Поздравляем, Вы справились!')

Поздравляем, Вы справились!
